In [1]:
# 测试 token_id 读取方法
import sys
import os
from pathlib import Path
import torch
# 添加项目根目录到 Python 路径
project_root = Path("/data/home/jlchen/code/UniVLA")
sys.path.append(str(project_root))

# 导入必要的库
from transformers import AutoTokenizer
from prismatic.models.vlms import PrismaticVLM
from latent_action_model.core.lam_model import LatentLAMModel
from dataclasses import dataclass
from typing import Optional

print("✅ 导入库成功")


/data/home/jlchen/miniconda3/envs/vla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-10 17:29:55.586033: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-10 17:29:55.619341: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-10 17:29:55.619365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-10 17

✅ 导入库成功


In [2]:
# 定义测试配置类
@dataclass
class TestConfig:
    """测试配置类，模拟训练配置"""
    # 模型配置
    model_id: str = 'OpenGVLab/InternVL3_5-1B-HF'
    hf_cache_dir: Optional[Path] = None
    
    # LAM 模型参数
    dim: int = 1024
    enc_layers: int = 6
    codebook_size: int = 16  # 码本大小，用于生成动作 token
    code_dim: int = 256
    dec_layers: int = 6
    dec_self_heads: int = 4
    dec_cross_heads: int = 4
    dropout: float = 0.1
    num_queries: int = 4
    
    # 其他参数
    hf_token: Optional[str] = None

# 创建测试配置实例
cfg = TestConfig()
print(f"✅ 测试配置创建成功，codebook_size = {cfg.codebook_size}")


✅ 测试配置创建成功，codebook_size = 16


In [3]:
# 初始化 VLM 模型和 tokenizer
print("🔄 正在加载 VLM 模型和 tokenizer...")

# 检查 PyTorch 版本
import torch
print(f"📋 PyTorch 版本: {torch.__version__}")

try:
    # 加载 VLM 模型
    vlm = PrismaticVLM.from_pretrained(
        cfg.model_id,
        cache_dir=cfg.hf_cache_dir,
        trust_remote_code=True,
        device_map="cuda",
    )
    
    # 获取 tokenizer
    tokenizer = vlm.tokenizer
    
    print(f"✅ VLM 模型加载成功: {cfg.model_id}")
    print(f"✅ Tokenizer 类型: {type(tokenizer)}")
    print(f"✅ 原始词汇表大小: {len(tokenizer)}")
    
except Exception as e:
    print(f"❌ 模型加载失败: {e}")
    print(f"🔍 错误类型: {type(e).__name__}")
    


🔄 正在加载 VLM 模型和 tokenizer...
📋 PyTorch 版本: 2.2.0+cu121
✅ VLM 模型加载成功: OpenGVLab/InternVL3_5-1B-HF
✅ Tokenizer 类型: <class 'transformers.models.qwen2.tokenization_qwen2_fast.Qwen2TokenizerFast'>
✅ 原始词汇表大小: 151679


In [4]:
# 测试添加特殊 token 的方法
print("🔄 测试添加动作 token...")

# 生成动作 token 列表
act_tokens = [f"<ACT_{i}>" for i in range(cfg.codebook_size)]
print(f"📝 生成的动作 token: {act_tokens}")

# 添加特殊 token 到 tokenizer
special_tokens_dict = {'additional_special_tokens': act_tokens}

try:
    num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
    print(f"✅ 成功添加 {num_added_toks} 个特殊 token")
    print(f"✅ 更新后词汇表大小: {len(tokenizer)}")
except Exception as e:
    print(f"❌ 添加特殊 token 失败: {e}")
    num_added_toks = 0


🔄 测试添加动作 token...
📝 生成的动作 token: ['<ACT_0>', '<ACT_1>', '<ACT_2>', '<ACT_3>', '<ACT_4>', '<ACT_5>', '<ACT_6>', '<ACT_7>', '<ACT_8>', '<ACT_9>', '<ACT_10>', '<ACT_11>', '<ACT_12>', '<ACT_13>', '<ACT_14>', '<ACT_15>']
✅ 成功添加 16 个特殊 token
✅ 更新后词汇表大小: 151695


In [5]:
# 测试核心功能：解析动作 token 的 ID 范围
print("🔄 测试动作 token ID 解析...")

# 这是训练脚本中的核心代码
act_tokens = [f"<ACT_{i}>" for i in range(cfg.codebook_size)]
act_ids = tokenizer.convert_tokens_to_ids(act_tokens)

print(f"📝 动作 token 列表: {act_tokens}")
print(f"🔢 对应的 token ID: {act_ids}")

# 计算 action_token_begin_id
action_token_begin_id = min(act_ids)
print(f"🎯 action_token_begin_id = {action_token_begin_id}")
print(f"📊 ID 范围: {min(act_ids)} - {max(act_ids)}")

# 验证所有 token 都能正确转换
print("\n🔍 详细验证:")
for i, (token, token_id) in enumerate(zip(act_tokens, act_ids)):
    print(f"  {i:2d}: {token:8s} -> ID: {token_id:4d}")
    
    # 验证反向转换
    decoded_token = tokenizer.convert_ids_to_tokens(token_id)
    print(f"      反向转换: ID {token_id:4d} -> {decoded_token}")
    
    # 检查是否一致
    if token == decoded_token:
        print(f"      ✅ 转换正确")
    else:
        print(f"      ❌ 转换错误！")
    print()


🔄 测试动作 token ID 解析...
📝 动作 token 列表: ['<ACT_0>', '<ACT_1>', '<ACT_2>', '<ACT_3>', '<ACT_4>', '<ACT_5>', '<ACT_6>', '<ACT_7>', '<ACT_8>', '<ACT_9>', '<ACT_10>', '<ACT_11>', '<ACT_12>', '<ACT_13>', '<ACT_14>', '<ACT_15>']
🔢 对应的 token ID: [151679, 151680, 151681, 151682, 151683, 151684, 151685, 151686, 151687, 151688, 151689, 151690, 151691, 151692, 151693, 151694]
🎯 action_token_begin_id = 151679
📊 ID 范围: 151679 - 151694

🔍 详细验证:
   0: <ACT_0>  -> ID: 151679
      反向转换: ID 151679 -> <ACT_0>
      ✅ 转换正确

   1: <ACT_1>  -> ID: 151680
      反向转换: ID 151680 -> <ACT_1>
      ✅ 转换正确

   2: <ACT_2>  -> ID: 151681
      反向转换: ID 151681 -> <ACT_2>
      ✅ 转换正确

   3: <ACT_3>  -> ID: 151682
      反向转换: ID 151682 -> <ACT_3>
      ✅ 转换正确

   4: <ACT_4>  -> ID: 151683
      反向转换: ID 151683 -> <ACT_4>
      ✅ 转换正确

   5: <ACT_5>  -> ID: 151684
      反向转换: ID 151684 -> <ACT_5>
      ✅ 转换正确

   6: <ACT_6>  -> ID: 151685
      反向转换: ID 151685 -> <ACT_6>
      ✅ 转换正确

   7: <ACT_7>  -> ID: 151686
      反

In [ ]:
print(tokenizer.eos_token_id)
print(tokenizer.image_token_id)

151645


AttributeError: Qwen2TokenizerFast has no attribute image_token_id

: 